In [101]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import os
import sys
import json
import logging

import torch
import transformers

from rich.console import Console
from rich.theme import Theme
from rich.markup import escape

from src.utils import logging_utils

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3,4,5,6"

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)

def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)
logger = logging.getLogger(__name__)
logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(f"{transformers.__version__=}")

2026-05-25 10:35:37 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-05-25 10:35:37 __main__ INFO     transformers.__version__='5.3.0'


In [3]:
from src.models import ModelandTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"

mt = ModelandTokenizer(
    model_key=MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print(f"n_layer={mt.n_layer}, n_embd={mt.n_embd}, n_heads={mt.model.config.num_attention_heads}")

2026-05-25 10:35:41 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-05-25 10:35:41 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-05-25 10:35:41 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7feec78a47a0>
2026-05-25 10:35:41 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7feec7e04cd0> server_hostname='huggingface.co' timeout=10
2026-05-25 10:35:41 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7feec856fe60>
2026-05-25 10:35:41 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-05-25 10:35:41 httpcore.http11 DEBUG    send_request_headers.complete
2026-05-25 10:35:41 http

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-05-25 10:36:04 httpcore.connection DEBUG    close.started
2026-05-25 10:36:04 httpcore.connection DEBUG    close.complete
2026-05-25 10:36:04 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-05-25 10:36:04 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7feec84f27b0>
2026-05-25 10:36:04 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7feec7e04cd0> server_hostname='huggingface.co' timeout=10
2026-05-25 10:36:04 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7feec6278380>
2026-05-25 10:36:04 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-05-25 10:36:04 httpcore.http11 DEBUG    send_request_headers.complete
2026-05-25 10:36:04 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

## Dataset

In [102]:
from src.goals.concept_data import GoalConceptDataset

DATA_PATH = "../data/goal_concepts/iterative_vs_recursive.json"
ds = GoalConceptDataset.load(DATA_PATH)

by_lang: dict[str, int] = {}
for s in ds:
    for lang in s.languages:
        by_lang[lang] = by_lang.get(lang, 0) + 1

print(f"Loaded {len(ds)} problems. Stubs per language: {by_lang}")
for s in ds:
    langs = ", ".join(s.languages)
    print(f"  [{langs:14s}] {s.function_name}")

Loaded 25 problems. Stubs per language: {'python': 25, 'cpp': 25, 'java': 25, 'rust': 25}
  [python, cpp, java, rust] flatten_list
  [python, cpp, java, rust] sum_nested
  [python, cpp, java, rust] max_depth
  [python, cpp, java, rust] product_list
  [python, cpp, java, rust] sum_digits
  [python, cpp, java, rust] digital_root
  [python, cpp, java, rust] compress
  [python, cpp, java, rust] reverse_list
  [python, cpp, java, rust] is_palindrome_list
  [python, cpp, java, rust] count_nested
  [python, cpp, java, rust] fibonacci
  [python, cpp, java, rust] triangle_number
  [python, cpp, java, rust] sum_squares
  [python, cpp, java, rust] count_even
  [python, cpp, java, rust] remove_falsy
  [python, cpp, java, rust] interleave_none
  [python, cpp, java, rust] alternating_sum
  [python, cpp, java, rust] duplicate_each
  [python, cpp, java, rust] running_max
  [python, cpp, java, rust] pairwise_sums
  [python, cpp, java, rust] factorial
  [python, cpp, java, rust] count_set_bits
  [python

## Strategy labeler

AST-based classifier. Returns `"recursive"`, `"iterative"`, or `"Unsure"`.

- **Recursive**: body contains a `Call` whose callee name matches `function_name` (direct self-call).
- **Iterative**: body contains a `For` or `While` loop (and no self-call).
- **Unsure**: neither, both, or parse error (e.g. one-liner via comprehension/slice/builtin, or mixed strategy).

Note: list comprehensions (`ListComp`) are NOT treated as iterative here — they fall into `"Unsure"`. This is intentional for the first pass; adapt as needed.

In [103]:
import ast
import re


# ── Python labeler (AST-based) ────────────────────────────────────────────────

def label_strategy_python(code: str, function_name: str) -> str:
    """AST-based strategy classifier for Python.

    Covers two recursion patterns:
      1. Direct self-call: outer function calls itself by name.
      2. Nested-helper recursion: an inner function defined inside the outer
         function calls itself by name (e.g. a ``_helper`` that recurses).

    Priority: recursive > iterative > Unsure.
    """
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return "Unsure"

    func_def = None
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == function_name:
            func_def = node
            break

    if func_def is None:
        return "Unsure"

    has_recursion = False
    has_loop = False

    for node in ast.walk(func_def):
        if isinstance(node, (ast.For, ast.While)):
            has_loop = True
        if isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id == function_name:
                has_recursion = True
        # Nested helper that recurses on itself
        if (
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name != function_name
        ):
            helper_name = node.name
            for inner in ast.walk(node):
                if isinstance(inner, ast.Call):
                    if isinstance(inner.func, ast.Name) and inner.func.id == helper_name:
                        has_recursion = True

    if has_recursion:
        return "recursive"
    if has_loop:
        return "iterative"
    return "Unsure"


# ── Heuristic labeler (regex-based) ──────────────────────────────────────────
# Supports C++, Java, Rust, and any language with C-style {}-delimited bodies.
# Java and C++ work identically (for/while loops, camelCase function names).
# Rust adds `loop {}` as a third loop construct.

_LOOP_PATTERNS: dict[str, str] = {
    "rust": r"\b(for|while|loop)\b",
}
_DEFAULT_LOOP_PATTERN = r"\b(for|while)\b"


def label_strategy_heuristic(code: str, function_name: str, language: str = "") -> str:
    """Regex-based strategy classifier for C++, Java, Rust, and similar.

    Extracts the function body (text after the first '{') to avoid matching
    the function name in the signature or preceding comment.  Strips line and
    block comments before searching.

    Recursion : function_name( appears in the body.
    Iterative : a loop keyword appears in the body.
    Priority  : recursive > iterative > Unsure.
    """
    brace_idx = code.find("{")
    body = code[brace_idx + 1:] if brace_idx != -1 else code

    body = re.sub(r"//[^\n]*", "", body)
    body = re.sub(r"/\*.*?\*/", "", body, flags=re.DOTALL)

    loop_pat = _LOOP_PATTERNS.get(language, _DEFAULT_LOOP_PATTERN)

    has_recursion = bool(re.search(rf"\b{re.escape(function_name)}\s*\(", body))
    has_loop      = bool(re.search(loop_pat, body))

    if has_recursion:
        return "recursive"
    if has_loop:
        return "iterative"
    return "Unsure"


# ── Unified dispatcher ────────────────────────────────────────────────────────

def label_strategy(code: str, function_name: str, language: str = "python") -> str:
    """Classify iterative vs recursive strategy for any supported language.

    - python  : AST-based (robust, handles nested helpers)
    - cpp     : regex heuristic (for/while loops)
    - java    : regex heuristic (for/while loops; same as cpp)
    - rust    : regex heuristic (for/while/loop keywords)
    - others  : regex heuristic with default for/while pattern

    Returns: 'recursive' | 'iterative' | 'Unsure'
    """
    if language == "python":
        return label_strategy_python(code, function_name)
    return label_strategy_heuristic(code, function_name, language=language)


# ── Smoke tests ───────────────────────────────────────────────────────────────

_py_direct_rec = """
def flatten_list(lst):
    if not lst: return []
    head, *tail = lst
    return (flatten_list(head) if isinstance(head, list) else [head]) + flatten_list(tail)
"""
_py_iterative = """
def flatten_list(lst):
    stack, result = [lst], []
    while stack:
        item = stack.pop()
        if isinstance(item, list): stack.extend(reversed(item))
        else: result.append(item)
    return result
"""
_py_nested_helper = """
def flatten_list(lst):
    out = []
    def _go(x):
        for item in x:
            if isinstance(item, list): _go(item)
            else: out.append(item)
    _go(lst)
    return out
"""
_py_oneliner = "def flatten_list(lst): return [x for sub in lst for x in sub]"

_cpp_rec  = "int factorial(int n) {\n    if (n <= 1) return 1;\n    return n * factorial(n - 1);\n}"
_cpp_iter = "int factorial(int n) {\n    int r = 1;\n    for (int i = 2; i <= n; ++i) r *= i;\n    return r;\n}"
_cpp_unsure = "int factorial(int n) { return (n <= 1) ? 1 : n * (n-1); }"

_rust_iter_for   = "fn factorial(n: u64) -> u64 {\n    let mut r = 1;\n    for i in 1..=n { r *= i; }\n    r\n}"
_rust_iter_loop  = "fn factorial(n: u64) -> u64 {\n    let (mut r, mut i) = (1, 1);\n    loop { if i > n { break; } r *= i; i += 1; }\n    r\n}"
_rust_rec        = "fn factorial(n: u64) -> u64 {\n    if n == 0 { 1 } else { n * factorial(n - 1) }\n}"

_java_iter = "int countEven(List<Integer> lst) {\n    int c = 0;\n    for (int x : lst) if (x % 2 == 0) c++;\n    return c;\n}"
_java_rec  = "int countEven(List<Integer> lst) {\n    if (lst.isEmpty()) return 0;\n    return (lst.get(0) % 2 == 0 ? 1 : 0) + countEven(lst.subList(1, lst.size()));\n}"

assert label_strategy(_py_direct_rec,    "flatten_list", "python") == "recursive"
assert label_strategy(_py_iterative,     "flatten_list", "python") == "iterative"
assert label_strategy(_py_nested_helper, "flatten_list", "python") == "recursive"
assert label_strategy(_py_oneliner,      "flatten_list", "python") == "Unsure"
assert label_strategy(_cpp_rec,          "factorial",    "cpp")    == "recursive"
assert label_strategy(_cpp_iter,         "factorial",    "cpp")    == "iterative"
assert label_strategy(_cpp_unsure,       "factorial",    "cpp")    == "Unsure"
assert label_strategy(_rust_iter_for,    "factorial",    "rust")   == "iterative"
assert label_strategy(_rust_iter_loop,   "factorial",    "rust")   == "iterative"   # loop {}
assert label_strategy(_rust_rec,         "factorial",    "rust")   == "recursive"
assert label_strategy(_java_iter,        "countEven",    "java")   == "iterative"
assert label_strategy(_java_rec,         "countEven",    "java")   == "recursive"
print("All label_strategy smoke tests passed")

All label_strategy smoke tests passed


In [104]:
code = '''def quick_sort(arr):
    """Sorts an array using the quick sort algorithm."""
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quick_sort(left) + middle + quick_sort(right)'''

label_strategy(code, "quick_sort")

'recursive'

## Generation helpers

In [105]:
from src.goals.concept_data import GoalConceptSample, StrategyDirective
from src.goals.functional import apply_chat_template
from src.tokens import prepare_input

_SYSTEM_PROMPTS = {
    "python": "You are a helpful assistant proficient in Python. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.",
    "cpp": "You are a helpful assistant proficient in C++. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.",
    "java": "You are a helpful assistant proficient in Java. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.",
    "haskell": "You are a helpful assistant proficient in Haskell. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.",
    "rust": "You are a helpful assistant proficient in Rust. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.",
}


def build_prompt(
    mt,
    sample: GoalConceptSample,
    lang: str = "python",
    add_docstring: bool = True,
    without_comments: bool = False,
    preferred_strategy: StrategyDirective | None = None,
) -> str:
    """Build a chat prompt for *sample* in the given language.

    Uses apply_chat_template so the model sees:
        [system] language-specific system prompt
        [user]   sample.full_instruction
        [assistant] <code>\\n<stub>
    where the stub optionally includes the docstring in the idiomatic style
    for *lang* (controlled by add_docstring and sample.without_comments).
    """
    stub = sample.get_code_stub(lang, add_docstring=add_docstring)
    system_prompt = _SYSTEM_PROMPTS.get(lang, _SYSTEM_PROMPTS["python"])
    comment_config_cache = sample.without_comments
    strategy_config_cache = sample.preferred_strategy
    sample.without_comments = without_comments  # override for prompt construction
    sample.preferred_strategy = preferred_strategy  # override for prompt construction

    prompt = apply_chat_template(
        mt,
        user_query=sample.full_instruction,
        incomplete_code=stub,
        system_prompt=system_prompt,
    )

    sample.without_comments = comment_config_cache  # restore original config
    sample.preferred_strategy = strategy_config_cache  # restore original config
    return prompt


@torch.inference_mode()
def sample_completions(
    mt,
    prompt: str,
    n: int = 10,
    temperature: float = 0.8,
    max_new_tokens: int = 300,
    stop_tag: str = "</code>",
) -> list[str]:
    """Generate n independent completions from prompt via sampling.

    Returns decoded continuation strings (prompt stripped, stop_tag stripped).
    """
    inputs = prepare_input(prompts=prompt, tokenizer=mt)
    input_ids = inputs["input_ids"].to(mt._model.device).expand(n, -1)
    attention_mask = inputs["attention_mask"].to(mt._model.device).expand(n, -1)
    prompt_len = input_ids.shape[1]

    outputs = mt._model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=mt.tokenizer.eos_token_id,
    )

    results = []
    for out in outputs:
        text = mt.tokenizer.decode(out[prompt_len:], skip_special_tokens=True)
        if stop_tag in text:
            text = text[: text.index(stop_tag)]
        results.append(text.rstrip())
    return results


sample = ds[0]
check_prompt = build_prompt(
    mt,
    sample,
    lang="python",
    add_docstring=False,
    without_comments=True,
    preferred_strategy="recursive",
)
print(check_prompt)

<|im_start|>system
You are a helpful assistant proficient in Python. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function that flattens a nested list of integers into a single flat list, preserving order. Implement the function recursively. Write your code without any comments or explanations, just the implementation.<|im_end|>
<|im_start|>assistant
<code>
def flatten_list(lst: list) -> list:


In [106]:
# prompt = '''<|im_start|>system
# You are a helpful assistant proficient in Python. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
# <|im_start|>user
# Write a function that flattens a nested list of integers into a single flat list, preserving order. Write your code without any comments or explanations, just the implementation.<|im_end|>
# <|im_start|>assistant
# <code>
# def flatten_list(lst: list) -> list:'''

# completions = sample_completions(mt, prompt, n=20)
# for i, c in enumerate(completions):
#     print(f"Completion {i+1}:\n\"{c}\"\n{'-'*40}")

## Run pipeline

For each problem: sample `N_COMPLETIONS` completions, reconstruct the full function, label strategy.

In [107]:
from tqdm.auto import tqdm

LANG = [
    "python",
    "cpp",
    "java",
    "rust",
]  # language to sample in; must be in sample.languages
N_COMPLETIONS = 20
TEMPERATURE = 0.7

# results[i] = {
#   'sample':      GoalConceptSample
#   'lang':        language used
#   'completions': list of raw generated strings (continuation only)
#   'full_codes':  list of stub + completion strings
#   'labels':      list of label_strategy outputs
# }
results = []

for lang in LANG:
    logger.info("LANGUAGE: %s", lang)
    for sample in tqdm(ds.filter_language(lang), desc="problems"):
        prompt = build_prompt(
            mt,
            sample,
            lang=lang,
            without_comments=True,
            add_docstring=False,
            preferred_strategy="recursive",
        )
        completions = sample_completions(
            mt, prompt, n=N_COMPLETIONS, temperature=TEMPERATURE
        )

        stub = sample.get_code_stub(lang, add_docstring=False)
        full_codes = [stub + c for c in completions]
        fn_name = sample.get_function_name(lang)
        labels = [label_strategy(code, fn_name, language=lang) for code in full_codes]

        results.append(
            {
                "sample": sample,
                "prompt": prompt,
                "stub": stub,
                "lang": lang,
                "completions": completions,
                "full_codes": full_codes,
                "labels": labels,
            }
        )

print("Done.")

2026-05-25 14:48:06 __main__ INFO     LANGUAGE: python


problems:   0%|          | 0/25 [00:00<?, ?it/s]

2026-05-25 14:49:59 __main__ INFO     LANGUAGE: cpp


problems:   0%|          | 0/25 [00:00<?, ?it/s]

2026-05-25 14:52:51 __main__ INFO     LANGUAGE: java


problems:   0%|          | 0/25 [00:00<?, ?it/s]

2026-05-25 14:55:36 __main__ INFO     LANGUAGE: rust


problems:   0%|          | 0/25 [00:00<?, ?it/s]

Done.


## Strategy distribution per problem

In [10]:
from collections import Counter
from rich.table import Table

table = Table(title="Strategy distribution per problem", show_lines=True)
table.add_column("function", style="cyan")
table.add_column("lang", style="dim")
table.add_column("recursive", justify="right")
table.add_column("iterative", justify="right")
table.add_column("Unsure", justify="right")
table.add_column("total", justify="right")

recursive_approaches = []
threshold = .7

for r in results:
    c = Counter(r["labels"])
    n_total = len(r["labels"])
    table.add_row(
        r["sample"].function_name,
        r["lang"],
        str(c.get("recursive", 0)),
        str(c.get("iterative", 0)),
        str(c.get("Unsure", 0)),
        str(n_total),
    )
    total = c.get("recursive", 0) + c.get("iterative", 0)  + c.get("Unsure", 0)
    if c.get("recursive", 0) / total >= threshold:
        recursive_approaches.append((r))


console.print(table)

                   Strategy distribution per problem                    
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ function           ┃ lang   ┃ recursive ┃ iterative ┃ Unsure ┃ total ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ flatten_list       │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ sum_nested         │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ max_depth          │ python │         9 │         0 │     11 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ product_list       │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ sum_digits         │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ digital_root       │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ compress           │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ reverse_list       │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ is_palindrome_list │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ count_nested       │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ fibonacci          │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ triangle_number    │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ sum_squares        │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ count_even         │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ remove_falsy       │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ interleave_none    │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ alternating_sum    │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ duplicate_each     │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ running_max        │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ pairwise_sums      │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ factorial          │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ count_set_bits     │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ reverse_number     │ python │        13 │         0 │      7 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ collatz_steps      │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ gcd                │ python │        20 │         0 │      0 │    20 │
├────────────────────┼────────┼───────────┼───────────┼────────┼───────┤
│ flatten_list       │ cpp    │        20 │         0 │   

## Per-problem inspection

Browse completions for a specific problem to spot labeling errors or borderline cases.

In [11]:
INSPECT_FUNCTION = "gcd"   # change to inspect another

r_filtered = [x for x in results if x["sample"].function_name == INSPECT_FUNCTION]

for i, result in enumerate(r_filtered):
    sample = result["sample"]
    lang   = result["lang"]
    stub = result["stub"]
    prompt = result["prompt"]
    fn_name = sample.get_function_name(lang)
    logger.info(f"Inspecting sample #{i} for function '{INSPECT_FUNCTION}' in {lang}")
    print(f"Prompt:\n\"{prompt}\"\n{'-'*80}")
    recursive_count = iterative_count = unsure_count = 0

    for idx, completion in enumerate(result["completions"]):
        full_code = stub + completion
        assert full_code == result["full_codes"][idx], "Full code mismatch!"
        label = label_strategy(full_code, fn_name, language=lang)
        if label == "recursive":
            recursive_count += 1
        elif label == "iterative":
            iterative_count += 1
        else:
            unsure_count += 1
        rich_print(
            f"Sample #{idx} - Label: [bold][salmon1]{label}[/salmon1][/bold]\n"
            f"[white]{escape(stub)}[/white][bright_yellow]{escape(completion)}[/bright_yellow]\n"
        )

    print(f"Summary for {INSPECT_FUNCTION} ({lang}):")
    print(f"  Recursive: {recursive_count}")
    print(f"  Iterative: {iterative_count}")
    print(f"  Unsure:    {unsure_count}\n\n")

label_strategy(full_code, "flatten_list", language="cpp")

2026-05-24 14:09:06 __main__ INFO     Inspecting sample #0 for function 'gcd' in python
Prompt:
"<|im_start|>system
You are a helpful assistant proficient in Python. When completing code, ensure proper syntax and logical correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a function that returns the greatest common divisor of two non-negative integers a and b. Implement the function recursively. Write your code without any comments or explanations, just the implementation.<|im_end|>
<|im_start|>assistant
<code>
def gcd(a: int, b: int) -> int:"
--------------------------------------------------------------------------------


Sample #0 - Label: recursive
def gcd(a: int, b: int) -> int:  
    return a if b == 0 else gcd(b, a % b)

Sample #1 - Label: recursive
def gcd(a: int, b: int) -> int: return a if b == 0 else gcd(b, a % b)

Sample #2 - Label: recursive
def gcd(a: int, b: int) -> int:  
    return a if b == 0 else gcd(b, a % b)

Sample #3 - Label: recursive
def gcd(a: int, b: int) -> int:  
    if b == 0:  
        return a  
    return gcd(b, a % b)

Sample #4 - Label: recursive
def gcd(a: int, b: int) -> int: return a if b == 0 else gcd(b, a % b)

... (76 outputs left out)

Sample #18 - Label: recursive
pub fn gcd(a: i32, b: i32) -> i32 { if b == 0 { a } else { gcd(b, a % b) } }

Sample #19 - Label: recursive
pub fn gcd(a: i32, b: i32) -> i32 { if b == 0 { a } else { gcd(b, a % b) } }

Summary for gcd (rust):
  Recursive: 20
  Iterative: 0
  Unsure:    0




'Unsure'

In [12]:
print(full_code)

pub fn gcd(a: i32, b: i32) -> i32 { if b == 0 { a } else { gcd(b, a % b) } }


In [13]:
results

[{'sample': GoalConceptSample(function_name='flatten_list', concept='iterative_vs_recursive', instruction='Write a function that flattens a nested list of integers into a single flat list, preserving order.', unit_tests=[([1, [2, [3, 4]], 5], [1, 2, 3, 4, 5]), ([], []), ([1, 2, 3], [1, 2, 3]), ([[1, [2, [3]]]], [1, 2, 3]), ([[1], 2, [3, [4]]], [1, 2, 3, 4])], pivot_note='Fork is at function body start; recursive uses self-calls, iterative uses a stack/deque.', stubs={'python': 'def flatten_list(lst: list) -> list:\n    ', 'cpp': 'std::vector<int> flattenList(const std::vector<std::any>& lst) {\n    ', 'java': 'public static List<Integer> flattenList(List<Object> lst) {\n    ', 'rust': 'pub fn flatten_list(lst: &[serde_json::Value]) -> Vec<i32> {\n    '}, docstring='Return a flat list of all integers from the nested input list, preserving order.', without_comments=False, preferred_strategy=None, no_comment_str='Write your code without any comments or explanations, just the implementatio

## Analysis

In [ ]:
from src.functional import get_hs
from src.tokens import prepare_input

target_token_range = list(range(-15, 0))
layers = mt.layer_names
hs_by_results = []

for r in tqdm(recursive_approaches):
    prompt = r["prompt"]
    tokenized_prompt = prepare_input(prompts=prompt, tokenizer=mt)
    hs = get_hs(
        mt = mt,
        input = tokenized_prompt,
        locations = [(layer, token_idx) for layer in layers for token_idx in target_token_range],
        return_dict = True
    )
    hs_by_results.append((r, hs))

In [ ]:
from src.goals.head_loc_attn import HeadMask
from src.utils import env_utils

MASK_PATH = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    f"localization/{MODEL_NAME.split('/')[-1]}/head_mask.pt",
)

ckpt = torch.load(MASK_PATH, map_location="cpu")
print(f"Loaded checkpoint from {MASK_PATH}")
# print(f"  Trained for {ckpt['config']['n_epochs']} epochs  |  N={ckpt['config']['n_instances']} instances")
# print(f"  λ_sparse={ckpt['config']['lambda_sparse']}  lr={ckpt['config']['lr']}")

# Reconstruct mask object
LAYERS = ckpt["layers"]
n_layers_tested = len(LAYERS)
n_heads = ckpt["mask_values"].shape[1]
mask = HeadMask(n_layers_tested, n_heads)
with torch.no_grad():
    mask.values.copy_(ckpt["mask_values"])
    if mt.name.split("/")[-1] == "Qwen2.5-Coder-32B-Instruct":
        mask.values[0:25] = 0  # zero out early layers
        mask.values[56:] = 0  # zero out late layers
    if mt.name.split("/")[-1] == "Llama-3.3-70B-Instruct":
        mask.values[0:18] = 0  # zero out early layers
        mask.values[45:] = 0  # zero out late layers

threshold = 0.9
selected_heads = mask.selected_heads(LAYERS, threshold=threshold)
print(
    f"\n  Layers: {LAYERS[0]}–{LAYERS[-1]}  ({n_layers_tested} tested, {n_heads} heads each)"
)
print(f"  Active heads (mask > {threshold}): {len(selected_heads)}")
print(f"  Selected heads: {selected_heads}")

_n_lt = len(LAYERS)
_n_h = mt.config.num_attention_heads
binary_mask = HeadMask(_n_lt, _n_h)
with torch.no_grad():
    binary_mask.values.zero_()
    for _layer_idx, _head_idx in selected_heads:
        binary_mask.values[LAYERS.index(_layer_idx), _head_idx] = 1.0

print(
    f"Binary mask: {binary_mask.n_active()} active heads / {_n_lt * _n_h} total heads "
    f"({100 * binary_mask.n_active() / (_n_lt * _n_h):.2f}%)"
)

In [ ]:
from src.goals.head_loc_attn import get_o_proj_inputs_batch

prompts = [r["prompt"] for r in recursive_approaches]
batch_tokenized = prepare_input(prompts=prompts, tokenizer=mt)

_src_o_inp = get_o_proj_inputs_batch(
    mt = mt,
    prompts = prompts,
    layers = list(range(mt.n_layer)),
    token_positions = target_token_range,
)

In [ ]:
from src.goals.goalscope import _build_goalscope_prompt
from src.goals.goalscope import apply_goalscope, headpatched_generate

goalscope_prompt = _build_goalscope_prompt(mt)
print(goalscope_prompt)
goalscope_inputs = prepare_input(prompts=goalscope_prompt, tokenizer=mt)
debug_aggregated = {}

for token_idx in target_token_range:
    _src_o_inp_token = torch.stack([_src_o_inp[ly][:, token_idx] for ly in LAYERS], dim=0)
    # print(f"  Shape: {_src_o_inp_token.shape} (layers x batch x model_dim)")
    _src_o_aggregated = _src_o_inp_token.mean(dim=1)
    # print(f"  Aggregated shape: {_src_o_aggregated.shape} (layers x model_dim)")

    verbalization_head = headpatched_generate(
        mt=mt,
        dest_inputs = goalscope_inputs,
        src_o_proj_inps = _src_o_aggregated,
        mask = binary_mask,
        layers = list(range(mt.n_layer)),
        max_new_tokens = 64,
    )
    print(f"{token_idx=}, {verbalization_head=}")

In [ ]:
from src.goals.head_loc_attn import collect_head_steering_vecs

aggregated_o_proj_outs_batch = collect_head_steering_vecs(
    mt=mt,
    src_tokenized = batch_tokenized,
    selected_heads = selected_heads,
    token_positions = target_token_range,
)
aggregated_o_proj_outs_batch.shape

In [ ]:
aggregated_o_proj_outs = aggregated_o_proj_outs_batch.mean(dim=0)
aggregated_o_proj_outs.shape

In [ ]:
from src.functional import PatchSpec, generate_with_patch

STEER_LAYER_RANGE = (35, 46)

for token_idx in target_token_range:
    patches = [
        PatchSpec(
            location = (mt.layer_name_format.format(li), -1),
            patch = aggregated_o_proj_outs[token_idx] * .2,
            strategy = "add",
            on_pass = "all"
        )
        for li in range(*STEER_LAYER_RANGE)
    ]
    verbalization_steer = generate_with_patch(
        mt=mt,
        inputs = goalscope_inputs,
        patches = patches,
        max_new_tokens = 64,
        patch_interface = "baukit", 
        remove_prefix=True,
        do_sample=False,
    )[0]
    print(f"{token_idx=}, {verbalization_steer=}")

In [ ]:
TARGET_LAYER = 50

for token_idx in target_token_range:
    all_resid = torch.stack([
        hs[mt.layer_name_format.format(TARGET_LAYER), token_idx] for r, hs in hs_by_results
    ])
    h_aggregated = all_resid.mean(dim=0).unsqueeze(0)
    patches = [
        PatchSpec(
            location=(mt.layer_name_format.format(li), -1),
            patch=h_aggregated * .1,
            strategy="add",
            on_pass="all",
        )
        for li in range(*STEER_LAYER_RANGE)
    ]
    verbalization_hs = generate_with_patch(
        mt=mt,
        inputs = goalscope_inputs,
        patches = patches,
        max_new_tokens = 32,
        patch_interface = "baukit", 
        remove_prefix=True,
        do_sample=False,
    )[0]
    print(f"{token_idx=}, {verbalization_hs=}")

## Loading and labeling proactive generations

In [13]:
# ── 1. Load proactive-strategy generations from disk ────────────────────────
# Each JSON is one task: { function_name, prompt, generation, prompt_bias,
# prompt_token_ids, generation_token_ids, greedy: {...}, samples: [...] }.

import json
from collections import Counter
from pathlib import Path

import pandas as pd

from src.utils import env_utils

RESULTS_DIR = Path(
    os.path.join(
        env_utils.DEFAULT_RESULTS_DIR,
        "well-known-tasks/generations",
        mt.name.split("/")[-1]
    )
)  

records = []
for src_dir in sorted(p for p in RESULTS_DIR.iterdir() if p.is_dir()):
    for fp in sorted(src_dir.glob("*.json")):
        with open(fp) as f:
            records.append(json.load(f))

print(f"Loaded {len(records)} records from {RESULTS_DIR}")
print(f"Sources : {dict(Counter(r['source'] for r in records))}")
print(f"Keys    : {list(records[0].keys())}")


Loaded 293 records from /disk/u/arnab/Codes/Projects/goals/results/well-known-tasks/generations/Qwen2.5-Coder-32B-Instruct
Sources : {'custom': 176, 'humaneval': 73, 'mbpp': 44}
Keys    : ['function_name', 'source', 'prompt', 'generation', 'prompt_bias', 'prompt_token_ids', 'generation_token_ids', 'greedy', 'samples']


In [14]:
# ── 2. Summary: prompt_bias × greedy.strategy ───────────────────────────────
# Quick read on the dataset before any filtering.

summary = pd.DataFrame([
    {
        "function_name": r["function_name"],
        "prompt_bias":   r["prompt_bias"],
        "greedy_strategy": r["greedy"]["strategy"],
        "n_rec_calls":   len(r["greedy"]["recursive_calls"]),
        "code_tag_fb":   r["greedy"]["used_code_tag_fallback"],
        "ast_fb":        r["greedy"]["used_ast_fallback"],
    }
    for r in records
])

print("prompt_bias × greedy_strategy:")
print(pd.crosstab(summary["prompt_bias"], summary["greedy_strategy"]))
summary.sort_values(["prompt_bias", "greedy_strategy"])


prompt_bias × greedy_strategy:
greedy_strategy  Unsure  iterative  recursive
prompt_bias                                  
iterative             0          5          0
neutral              67        188         29
recursive             1          1          2


,function_name,prompt_bias,greedy_strategy,n_rec_calls,code_tag_fb,ast_fb
40,dfs,iterative,iterative,0,True,True
57,gcd_iterative,iterative,iterative,0,True,True
64,inorder_traversal,iterative,iterative,0,True,True
113,merge_sort_iterative,iterative,iterative,0,True,True
143,reverse_linked_list,iterative,iterative,0,True,True
...,...,...,...,...,...,...
289,str_to_list,neutral,recursive,1,True,True
11,binary_search_recursive,recursive,Unsure,0,True,True
8,bfs_recursive,recursive,iterative,0,True,True
65,inorder_traversal_recursive,recursive,recursive,2,True,True


In [15]:
r["greedy"]

{'extracted_code': 'Sorts an array using the quick sort algorithm."""\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[len(arr) // 2]\n    left = [x for x in arr if x < pivot]\n    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n    return quick_sort(left) + middle + quick_sort(right)\n</code>',
 'used_code_tag_fallback': True,
 'strategy': 'recursive',
 'has_recursion': True,
 'has_loop': False,
 'used_ast_fallback': True,
 'recursive_calls': [{'name': 'quick_sort',
   'name_char_span': [260, 270],
   'name_token_positions': [78, 79]},
  {'name': 'quick_sort',
   'name_char_span': [288, 298],
   'name_token_positions': [85, 86]}]}

In [7]:
# ── 3. Strict proactive-recursion filter ────────────────────────────────────
# Keep tasks where: (a) prompt is neutral (no recursive/iterative pressure)
# AND (b) greedy completion contains a recursive call.

proactive_recursive = [
    r for r in records
    if r["prompt_bias"] == "neutral" and r["greedy"]["has_recursion"]
]

print(f"Proactive recursive: {len(proactive_recursive)} / {len(records)} tasks\n")
for idx, r in enumerate(proactive_recursive):
    calls = r["greedy"]["recursive_calls"]
    names = ", ".join(sorted({c["name"] for c in calls}))
    print(f" {idx:2d}   {r['function_name']:30s}  n_calls={len(calls):2d}  helpers={names}")


Proactive recursive: 29 / 293 tasks

  0   ackermann                       n_calls= 3  helpers=ackermann
  1   combinations                    n_calls= 1  helpers=combinations
  2   count_and_say                   n_calls= 1  helpers=count_and_say
  3   count_leaves                    n_calls= 2  helpers=count_leaves
  4   count_tree_nodes                n_calls= 2  helpers=count_tree_nodes
  5   factorial                       n_calls= 1  helpers=factorial
  6   flatten_nested_list             n_calls= 1  helpers=flatten_nested_list
  7   invert_binary_tree              n_calls= 2  helpers=invert_binary_tree
  8   is_subtree                      n_calls= 2  helpers=is_subtree
  9   josephus_problem                n_calls= 1  helpers=josephus_problem
 10   lowest_common_ancestor          n_calls= 2  helpers=lowest_common_ancestor
 11   max_depth                       n_calls= 2  helpers=max_depth
 12   merge_sort                      n_calls= 2  helpers=merge_sort
 13   merge_two_trees

In [8]:
# ── 4. Preference-strength via sampled completions ──────────────────────────
# rec_rate = fraction of n_samples completions that contained a recursive call.
# High rec_rate ⇒ the model strongly prefers recursion for this task under
# sampling.  Tasks at rec_rate ≈ 1.0 with prompt_bias == "neutral" are the
# cleanest candidates for downstream intent extraction.

rate_rows = []
for r in records:
    s = r.get("samples", [])
    rate_rows.append({
        "function_name":   r["function_name"],
        "prompt_bias":     r["prompt_bias"],
        "greedy_strategy": r["greedy"]["strategy"],
        "rec_rate":        sum(x["has_recursion"] for x in s) / len(s) if s else float("nan"),
        "loop_rate":       sum(x["has_loop"]      for x in s) / len(s) if s else float("nan"),
        "n_samples":       len(s),
    })

rate_df = pd.DataFrame(rate_rows).sort_values("rec_rate", ascending=False)
rate_df


,function_name,prompt_bias,greedy_strategy,rec_rate,loop_rate,n_samples
48,factorial,neutral,recursive,1.0,0.0,10
53,flatten_nested_list,neutral,recursive,1.0,1.0,10
255,div_sum,neutral,recursive,1.0,0.4,10
0,ackermann,neutral,recursive,1.0,0.0,10
69,invert_binary_tree,neutral,recursive,1.0,0.0,10
...,...,...,...,...,...,...
99,longest_substring_no_repeat,neutral,iterative,0.0,1.0,10
98,longest_palindromic_substring,neutral,iterative,0.0,1.0,10
97,longest_increasing_subsequence_bs,neutral,iterative,0.0,1.0,10
96,longest_increasing_subsequence,neutral,iterative,0.0,1.0,10


In [11]:
# ── 5. Visualize one proactive-recursive completion ─────────────────────────
# Decode token-by-token and highlight the indices stored in
# greedy.recursive_calls[i].name_token_positions (positions are relative to
# generation_token_ids).

from rich import print as rich_print
from rich.text import Text

r = proactive_recursive[18]
gen_ids = r["generation_token_ids"]
call_positions = {p for c in r["greedy"]["recursive_calls"] for p in c["name_token_positions"]}

txt = Text()
txt.append(r["prompt"], style="dark_orange3")
for i, tid in enumerate(gen_ids):
    s = mt.tokenizer.decode([tid])
    if s == mt.tokenizer.eos_token:
        break
    if i in call_positions:
        txt.append(s, style="bold black on yellow")
    else:
        txt.append(s)

print(call_positions)
print(f"Task: {r['function_name']}  |  call positions: {sorted(call_positions)}\n")
rich_print(txt)


{86, 85, 78, 79}
Task: quick_sort  |  call positions: [78, 79, 85, 86]



<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Implement quick sort in Python. Write your code without any comments or explanations, just the 
implementation.<|im_end|>
<|im_start|>assistant
<code>
def quick_sort(arr):
    """Sorts an array using the quick sort algorithm."""
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quick_sort(left) + middle + quick_sort(right)
</code>

In [18]:
# ── 6. Prepare downstream-ready records ─────────────────────────────────────
# Compute *absolute* positions in the full sequence (prompt + generation),
# which is what attention/steering experiments will index into.
# Also sanity-check that the token at each absolute position decodes to the
# recorded function name.

downstream = []
for r in proactive_recursive:
    prompt_ids = r["prompt_token_ids"]
    gen_ids    = r["generation_token_ids"]
    full_ids   = prompt_ids + gen_ids
    prompt_len = len(prompt_ids)

    abs_calls = []
    for c in r["greedy"]["recursive_calls"]:
        abs_positions = [prompt_len + p for p in c["name_token_positions"]]
        abs_calls.append({"name": c["name"], "abs_positions": abs_positions})

    downstream.append({
        "function_name": r["function_name"],
        "full_token_ids": full_ids,
        "abs_calls": abs_calls,
    })

# Sanity check: decode each absolute position and confirm it lies inside the
# recorded function-name string.
for d in downstream:
    print(f"\n=== {d['function_name']} ===")
    for call in d["abs_calls"]:
        decoded = [mt.tokenizer.decode([d["full_token_ids"][p]]) for p in call["abs_positions"]]
        match = "".join(decoded).strip().startswith(call["name"][:3])
        print(f"  name={call['name']:15s}  abs={call['abs_positions']}  tokens={decoded}  ok={match}")



=== ackermann ===
  name=ackermann        abs=[109, 110]  tokens=[' ack', 'ermann']  ok=True
  name=ackermann        abs=[124, 125]  tokens=[' ack', 'ermann']  ok=True
  name=ackermann        abs=[131, 132]  tokens=[' ack', 'ermann']  ok=True

=== combinations ===
  name=combinations     abs=[138]  tokens=[' combinations']  ok=True

=== count_and_say ===
  name=count_and_say    abs=[109, 110, 111, 112]  tokens=[' count', '_and', '_s', 'ay']  ok=True

=== count_leaves ===
  name=count_leaves     abs=[165, 166, 167]  tokens=[' count', '_le', 'aves']  ok=True
  name=count_leaves     abs=[172, 173, 174]  tokens=[' count', '_le', 'aves']  ok=True

=== count_tree_nodes ===
  name=count_tree_nodes  abs=[117, 118, 119]  tokens=[' count', '_tree', '_nodes']  ok=True
  name=count_tree_nodes  abs=[124, 125, 126]  tokens=[' count', '_tree', '_nodes']  ok=True

=== factorial ===
  name=factorial        abs=[105]  tokens=[' factorial']  ok=True

=== flatten_nested_list ===
  name=flatten_nested_lis

In [20]:
from src.goals.head_loc_attn import HeadMask
from src.utils import env_utils

MASK_PATH = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    f"localization/{MODEL_NAME.split('/')[-1]}/head_mask.pt",
)

ckpt = torch.load(MASK_PATH, map_location="cpu")
print(f"Loaded checkpoint from {MASK_PATH}")
# print(f"  Trained for {ckpt['config']['n_epochs']} epochs  |  N={ckpt['config']['n_instances']} instances")
# print(f"  λ_sparse={ckpt['config']['lambda_sparse']}  lr={ckpt['config']['lr']}")

# Reconstruct mask object
LAYERS = ckpt["layers"]
n_layers_tested = len(LAYERS)
n_heads = ckpt["mask_values"].shape[1]
mask = HeadMask(n_layers_tested, n_heads)
with torch.no_grad():
    mask.values.copy_(ckpt["mask_values"])
    if mt.name.split("/")[-1] == "Qwen2.5-Coder-32B-Instruct":
        mask.values[0:25] = 0  # zero out early layers
        mask.values[56:] = 0  # zero out late layers
    if mt.name.split("/")[-1] == "Llama-3.3-70B-Instruct":
        mask.values[0:18] = 0  # zero out early layers
        mask.values[45:] = 0  # zero out late layers

threshold = 0.9
selected_heads = mask.selected_heads(LAYERS, threshold=threshold)
print(
    f"\n  Layers: {LAYERS[0]}–{LAYERS[-1]}  ({n_layers_tested} tested, {n_heads} heads each)"
)
print(f"  Active heads (mask > {threshold}): {len(selected_heads)}")
print(f"  Selected heads: {selected_heads}")

_n_lt = len(LAYERS)
_n_h = mt.config.num_attention_heads
binary_mask = HeadMask(_n_lt, _n_h)
with torch.no_grad():
    binary_mask.values.zero_()
    for _layer_idx, _head_idx in selected_heads:
        binary_mask.values[LAYERS.index(_layer_idx), _head_idx] = 1.0

print(
    f"Binary mask: {binary_mask.n_active()} active heads / {_n_lt * _n_h} total heads "
    f"({100 * binary_mask.n_active() / (_n_lt * _n_h):.2f}%)"
)

Loaded checkpoint from /disk/u/arnab/Codes/Projects/goals/results/localization/Qwen2.5-Coder-32B-Instruct/head_mask.pt

  Layers: 0–63  (64 tested, 40 heads each)
  Active heads (mask > 0.9): 122
  Selected heads: [(28, 13), (29, 17), (29, 18), (30, 8), (30, 25), (30, 35), (31, 18), (31, 32), (32, 3), (32, 13), (34, 25), (34, 30), (34, 33), (35, 13), (35, 39), (37, 13), (37, 15), (38, 0), (38, 2), (39, 4), (39, 8), (39, 9), (39, 14), (39, 31), (39, 32), (39, 34), (40, 2), (40, 8), (40, 11), (40, 14), (40, 32), (40, 34), (41, 6), (41, 9), (41, 12), (41, 13), (42, 39), (43, 5), (43, 8), (43, 13), (43, 30), (44, 12), (44, 13), (44, 14), (44, 18), (44, 38), (45, 15), (45, 17), (45, 18), (45, 21), (45, 22), (45, 31), (45, 32), (45, 33), (45, 34), (46, 1), (46, 4), (46, 7), (46, 8), (46, 9), (46, 18), (46, 19), (46, 20), (46, 25), (46, 26), (46, 35), (47, 5), (47, 6), (47, 7), (47, 8), (47, 18), (47, 19), (47, 23), (47, 30), (47, 32), (47, 35), (47, 39), (48, 25), (48, 26), (48, 28), (48, 29

In [81]:
from src.goals.goalscope import apply_goalscope

r = proactive_recursive[18]
prompt = r["prompt"]
gen = r["greedy"]["extracted_code"]

res = apply_goalscope(
    prompt + gen,
    mt=mt,
    selected_heads=selected_heads,
    strategy="replace", alpha=1.0,
    # strategy="additive", alpha=0.2,
    batch_size=16,
    # max_new_tokens=STEER_MAX_TOKENS,
    title=f"GoalScope (Head Patch) — {r['function_name']}"
)

GoalScope[replace]:   0%|          | 0/7 [00:00<?, ?it/s]

In [82]:
from src.functional import get_hs
from src.tokens import prepare_input
from tqdm.auto import tqdm
from src.goals.head_loc_attn import get_o_proj_inputs_batch, collect_head_steering_vecs

TARGET_OFFSET = list(range(-15, 0))

latents_by_results = []

for r in tqdm(proactive_recursive, desc="Processing proactive-recursive tasks"):
    prompt = r["prompt"]
    gen    = r["greedy"]["extracted_code"]
    full_prompt = prompt + gen
    tokenized_full = prepare_input(prompts=full_prompt, tokenizer=mt)

    recursive_call_start = [call["name_token_positions"][0] for call in r["greedy"]["recursive_calls"]]
    if len(recursive_call_start) == 0:
        continue

    recursive_call_start = len(r["prompt_token_ids"]) + min(recursive_call_start)
    target_token_range = [recursive_call_start + offset for offset in TARGET_OFFSET]

    hs = get_hs(
        mt = mt,
        input = tokenized_full,
        locations = [(layer, token_idx) for layer in mt.layer_names for token_idx in target_token_range],
        return_dict = True
    )

    src_o_inp = get_o_proj_inputs_batch(
        mt = mt,
        prompts = [full_prompt],
        layers = list(range(mt.n_layer)),
        token_positions = target_token_range,
    )

    src_o_proj_outs = collect_head_steering_vecs(
        mt=mt,
        src_tokenized = tokenized_full,
        selected_heads = selected_heads,
        token_positions = target_token_range,
    )

    latents_by_results.append({
        "result": r,
        "hs": hs,
        "src_o_inp": src_o_inp,
        "src_o_proj_outs": src_o_proj_outs,
        "token_positions": target_token_range,
    })

Processing proactive-recursive tasks:   0%|          | 0/29 [00:00<?, ?it/s]

In [83]:
latents = latents_by_results[0]
latents["src_o_proj_outs"].shape

torch.Size([1, 15, 5120])

In [85]:
from src.goals.goalscope import _build_goalscope_prompt
from src.goals.goalscope import apply_goalscope, headpatched_generate
from src.functional import PatchSpec, generate_with_patch

TARGET_SRC_LAYER = 55
STEER_LAYER_RANGE = (35, 46)

goalscope_prompt = _build_goalscope_prompt(mt)
print(goalscope_prompt)
goalscope_inputs = prepare_input(prompts=goalscope_prompt, tokenizer=mt)


for token_offset in range(-5, 0):
    print(f"\n=== Token offset {token_offset} ===")
    hs_by_results = []
    src_o_inp_by_results = []
    src_o_out_by_results = []
    for cached_latents in latents_by_results:
        r = cached_latents["result"]
        recursive_call_start = [
            call["name_token_positions"][0] for call in r["greedy"]["recursive_calls"]
        ]
        recursive_call_start = len(r["prompt_token_ids"]) + min(recursive_call_start)

        target_token_pos = recursive_call_start + token_offset
        token_idx = cached_latents["token_positions"].index(target_token_pos)
        # logger.debug(f"{r['function_name']} | {token_offset=}, {target_token_pos=}, {token_idx=}")

        hs_by_results.append(
            cached_latents["hs"][
                (mt.layer_name_format.format(TARGET_SRC_LAYER), target_token_pos)
            ]
        )
        src_o_inp_by_results.append(
            torch.stack(
                [
                    cached_latents["src_o_inp"][li][:, token_idx].squeeze()
                    for li in range(mt.n_layer)
                ], 
                dim = 0
            )
        )
        src_o_out_by_results.append(cached_latents["src_o_proj_outs"][:, token_idx])

    src_o_inp_by_results = torch.stack(src_o_inp_by_results, dim=0)
    # print(src_o_inp_by_results.shape)
    # raise NotImplementedError("Debugging - stop after one iteration")

    # h_aggregated = torch.stack(hs_by_results).mean(dim=0).unsqueeze(0)
    # patches = [
    #     PatchSpec(
    #         location=(mt.layer_name_format.format(li), -1),
    #         patch=h_aggregated * 0.1,
    #         strategy="add",
    #         on_pass="all",
    #     )
    #     for li in range(*STEER_LAYER_RANGE)
    # ]
    # verbalization_resid = generate_with_patch(
    #     mt=mt,
    #     inputs=goalscope_inputs,
    #     patches=patches,
    #     max_new_tokens=64,
    #     patch_interface="baukit",
    #     remove_prefix=True,
    #     do_sample=False,
    # )[0]
    # print(f"Verbalization (resid steer): {verbalization_resid}")

    o_inp_aggregated = src_o_inp_by_results.mean(dim=0)
    vergalization_head = headpatched_generate(
        mt=mt,
        dest_inputs=goalscope_inputs,
        src_o_proj_inps=o_inp_aggregated,
        mask=binary_mask,
        layers=list(range(mt.n_layer)),
        max_new_tokens=64,
    )
    print(f"Verbalization (head patch): {vergalization_head}")

    o_out_aggregated = torch.stack(src_o_out_by_results).mean(dim=0)
    patches = [
        PatchSpec(
            location=(mt.layer_name_format.format(li), -1),
            patch=o_out_aggregated * 0.2,
            strategy="add",
            on_pass="all",
        )
        for li in range(*STEER_LAYER_RANGE)
    ]
    verbalization_o_out = generate_with_patch(
        mt=mt,
        inputs=goalscope_inputs,
        patches=patches,
        max_new_tokens=64,
        patch_interface="baukit",
        remove_prefix=True,
        do_sample=False,
    )[0]
    print(f"Verbalization (o_out steer): {verbalization_o_out}")

<|im_start|>system
You are a helpful assistant proficient in programming.<|im_end|>
<|im_start|>user
What are you doing at this line of the code. Keep your answer as brief as possible.<|im_end|>
<|im_start|>assistant


=== Token offset -5 ===
Verbalization (head patch): I'm processing the current line to understand its context and functionality within the code.
Verbalization (o_out steer): Executing the current line's operation, which could be a function call, loop, or conditional statement, and then proceeding recursively or iteratively with subsequent lines.

=== Token offset -4 ===
Verbalization (head patch): I'm processing the current line to understand its context and then recursively checking its dependencies or children nodes.
Verbalization (o_out steer): Executing the current line's operation, which could be a function call, loop, or conditional statement, and then recursively doing the same for subsequent lines.

=== Token offset -3 ===
Verbalization (head patch): I'm processi

## Causal Validation

In [96]:
from src.goals.data import IncompleteCodeSample
from src.goals.functional import supports_chat_template, apply_chat_template
from src.goals.goalscope import greedy_continue

def get_prompt(sample, mt=mt):
    if supports_chat_template(mt):
        prompt = apply_chat_template(
            mt,
            user_query=sample.full_instruction.strip(),
            incomplete_code=sample.code,
        )
    else:
        prompt = sample.code

    return prompt

# test_sample = IncompleteCodeSample(
#     instruction="Write a C++ function that finds the greatest common divisor of two numbers. Implement it from scratch without using built-in functions.",
#     code='int gcd(int x, int y) {',
#     unit_tests=[],
#     function_name="gcd",
#     prefills=[],
#     augmentation=None,
# )
# test_sample.without_comments = True

test_sample = IncompleteCodeSample(
    instruction="Write a C++ function that sorts a list of numbers in ascending order. Implement it from scratch without using built-in sorting functions.",
    code='void sort(std::vector<int>& numbers) {',
    unit_tests=[],
    function_name="sort",
    prefills=[],
    augmentation=None,
)
test_sample.without_comments = True

# test_sample = IncompleteCodeSample(
#     instruction="Write a C++ function that reverses the characters of a string in place. Implement it from scratch without using built-in functions.",
#     code='void reverse_string(std::string& s) {',
#     function_name="reverse_string",
# )
# test_sample.without_comments = True

# test_sample = IncompleteCodeSample(
#     instruction="Write a C++ function that computes the sum of the decimal digits of a non-negative integer. Implement it from scratch without using built-in functions.",
#     code='int sum_of_digits(int n) {',
#     function_name="sum_of_digits",
# )
# test_sample.without_comments = True

# test_sample = IncompleteCodeSample(
#     instruction="Write a C++ function that converts a non-negative integer to its binary representation as a string. Implement it from scratch without using built-in functions.",
#     code='std::string decimal_to_binary(int n) {',
#     function_name="decimal_to_binary",
# )
# test_sample.without_comments = True

# test_sample = IncompleteCodeSample(
#     instruction="Write a C++ function that computes base raised to the power exp for non-negative integer exponents. Implement it from scratch without using built-in functions.",
#     code='long long power(int base, int exp) {',
#     function_name="power",
# )
# test_sample.without_comments = True

# test_sample = IncompleteCodeSample(
#     instruction="Write a C++ function that counts how many times a target value appears in a list of integers. Implement it from scratch without using built-in functions.",
#     code='int count_occurrences(std::vector<int>& numbers, int target) {',
#     function_name="count_occurrences",
# )
# test_sample.without_comments = True

# test_sample = IncompleteCodeSample(
#     instruction="Write a C++ function that checks whether a string reads the same forwards and backwards. Implement it from scratch without using built-in functions.",
#     code='bool is_palindrome(const std::string& s) {',
#     function_name="is_palindrome",
# )
# test_sample.without_comments = True

# test_sample = IncompleteCodeSample(
#     instruction="Write a C++ function that finds the maximum value in a non-empty list of integers. Implement it from scratch without using built-in functions.",
#     code='int find_max(std::vector<int>& numbers) {',
#     function_name="find_max",
# )
# test_sample.without_comments = True

# test_sample = IncompleteCodeSample(
#     instruction="Write a C++ function that reverses the digits of a non-negative integer. Implement it from scratch without using built-in functions.",
#     code='int reverse_integer(int n) {',
#     function_name="reverse_integer",
# )
# test_sample.without_comments = True

# test_sample = IncompleteCodeSample(
#     instruction="Write a C++ function that checks whether a number is present in a list of integers sorted in ascending order. The time complexity should be O(log n). Implement it from scratch without using built-in functions.",
#     code='bool search(std::vector<int>& numbers, int target) {',
#     function_name="search",
# )
# test_sample.without_comments = True

test_prompt = get_prompt(test_sample, mt)

test_gen = greedy_continue(
    mt,
    **prepare_input(prompts=test_prompt, tokenizer=mt),
    max_new_tokens=256,
)
rich_print(f"[white]{escape(test_prompt)}[/white][bright_yellow]{escape(test_gen)}[/bright_yellow]")

<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a C++ function that sorts a list of numbers in ascending order. Implement it from scratch without using 
built-in sorting functions. Write your code without any comments or explanations, just the 
implementation.<|im_end|>
<|im_start|>assistant
<code>
void sort(std::vector<int>& numbers) {  
    for (size_t i = 0; i < numbers.size(); ++i) {  
        for (size_t j = i + 1; j < numbers.size(); ++j) {  
            if (numbers[i] > numbers[j]) {  
                std::swap(numbers[i], numbers[j]);  
            }  
        }  
    }  
}
</code>

In [97]:
from src.goals.goalscope import apply_goalscope

res = apply_goalscope(
    test_prompt + test_gen,
    mt=mt,
    selected_heads=selected_heads,
    strategy="replace",
    batch_size=16,
    alpha=1.0,
    # max_new_tokens=STEER_MAX_TOKENS,
    title=f"GoalScope (Head Patch) — {test_sample.function_name}"
)


GoalScope[replace]:   0%|          | 0/6 [00:00<?, ?it/s]

In [99]:
from src.goals.goalscope import headpatched_generate_batch

# ── Surgical patched generation ───────────────────────────────────────────────

# TARGET_TOKEN_IDX = -8
# _src_o_inp_token = torch.stack([_src_o_inp[ly][:, TARGET_TOKEN_IDX] for ly in LAYERS], dim=0)
# _src_o_aggregated = _src_o_inp_token.mean(dim=1).unsqueeze(0) * 1.0

TARGET_TOKEN_OFFSET = -1

src_o_inp_by_results = []
src_o_out_by_results = []

for cached_latents in latents_by_results:
    r = cached_latents["result"]
    recursive_call_start = [
        call["name_token_positions"][0] for call in r["greedy"]["recursive_calls"]
    ]
    recursive_call_start = len(r["prompt_token_ids"]) + min(recursive_call_start)

    target_token_pos = recursive_call_start + TARGET_TOKEN_OFFSET
    token_idx = cached_latents["token_positions"].index(target_token_pos)
    # logger.debug(f"{r['function_name']} | {TARGET_TOKEN_OFFSET=}, {target_token_pos=}, {token_idx=}")

    src_o_inp_by_results.append(
        torch.stack(
            [
                cached_latents["src_o_inp"][li][:, token_idx].squeeze()
                for li in range(mt.n_layer)
            ], 
            dim = 0
        )
    )
    src_o_out_by_results.append(cached_latents["src_o_proj_outs"][:, token_idx])

_src_o_inp_token = torch.stack(src_o_inp_by_results, dim=0)
_src_o_aggregated = _src_o_inp_token.mean(dim=0).unsqueeze(0) * 2.0

print(_src_o_aggregated.shape)

verbalization_head = headpatched_generate(
    mt=mt,
    dest_inputs=goalscope_inputs,
    src_o_proj_inps=_src_o_aggregated,
    mask=binary_mask,
    layers=list(range(mt.n_layer)),
    max_new_tokens=64,
)
print(f"{verbalization_head=}")

headpatched_continuation = headpatched_generate_batch(
    mt=mt,
    dest_inputs=prepare_input(prompts=test_prompt, tokenizer=mt),
    src_o_proj_inps_batch=_src_o_aggregated,
    mask=binary_mask,
    layers=LAYERS,
    max_new_tokens=256,
    n_steer_tokens=20,
    # patch_input_positions=[-1],
)[0]

rich_print(
    f"[white]{escape(test_prompt)}[/white][bright_green]{escape(headpatched_continuation)}[/bright_green]"
)

torch.Size([1, 64, 5120])
verbalization_head="I'm checking the left subtree."


<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a C++ function that sorts a list of numbers in ascending order. Implement it from scratch without using 
built-in sorting functions. Write your code without any comments or explanations, just the 
implementation.<|im_end|>
<|im_start|>assistant
<code>
void sort(std::vector<int>& numbers) {  
    if (numbers.empty()) return;  
    std::vector<int> left(numbers.begin(), numbers.begin() + numbers.size() / 2);  
    std::vector<int> right(numbers.begin() + numbers.size() / 2, numbers.end());  
    sort(left);  
    sort(right);  
    numbers.clear();  
    while (!left.empty() && !right.empty()) {  
        if (left.front() <= right.front()) {  
            numbers.push_back(left.front());  
            left.erase(left.begin());  
        } else {  
            numbers.push_back(right.front());  
            right.erase(right.begin());  
        }  
    }  
    while (!left.empty()) {  
        numbers.push_back(left.front());  
        left.erase(left.begin());  
    }  
    while (!right.empty()) {  
        numbers.push_back(right.front());  
        right.erase(right.begin());  
    }  
}
</code>

In [100]:
# aggregated_o_proj_outs_target_token = collect_head_steering_vecs(
#     mt=mt,
#     src_tokenized=batch_tokenized,
#     selected_heads=selected_heads,
#     token_positions=[TARGET_TOKEN_IDX],
# )

# print(
#     f"{aggregated_o_proj_outs_target_token.shape=}",
#     f"{aggregated_o_proj_outs_batch[:, TARGET_TOKEN_IDX, :].shape=}",
# )

# print(
#     torch.allclose(
#         aggregated_o_proj_outs_target_token,
#         aggregated_o_proj_outs_batch[:, TARGET_TOKEN_IDX, :].unsqueeze(1),
#         atol=1e-2,
#     )
# )
# aggregated_o_proj_outs = aggregated_o_proj_outs_target_token.mean(dim=0)


aggregated_o_proj_outs = torch.stack(src_o_out_by_results).mean(dim=0)
patches = [
    PatchSpec(
        location=(mt.layer_name_format.format(li), -1),
        patch=aggregated_o_proj_outs.squeeze() * 0.2,
        strategy="add",
        on_pass="all",
    )
    for li in range(*STEER_LAYER_RANGE)
]

verbalization_steer = generate_with_patch(
    mt=mt,
    inputs=goalscope_inputs,
    patches=patches,
    max_new_tokens=64,
    patch_interface="baukit",
    remove_prefix=True,
    do_sample=False,
)[0]
print(f"{verbalization_steer=}")


# on first pass steer input token residual encodings 
# patches.extend(
#     PatchSpec(
#         location=(mt.layer_name_format.format(li), t_idx),
#         patch=aggregated_o_proj_outs.squeeze() * 0.2,
#         strategy="add",
#         on_pass="first",
#     )
#     for li in range(*STEER_LAYER_RANGE)
#     for t_idx in range(-3, -1)
# )

steered_continuation = generate_with_patch(
    mt=mt,
    inputs=prepare_input(prompts=test_prompt, tokenizer=mt),
    patches=patches,
    max_new_tokens=256,
    patch_interface="baukit",
    remove_prefix=True,
    do_sample=False,
    n_steer_tokens=20,
)[0]
rich_print(
    f"[white]{escape(test_prompt)}[/white][salmon1]{escape(steered_continuation)}[/salmon1]"
)

verbalization_steer="I'm analyzing the current line and recursively processing its sub-expressions or sub-statements."


<|im_start|>system
You are a helpful assistant proficient in programming. When completing code, ensure proper syntax and logical 
correctness. Make sure to have all codes inside the <code> </code> tags.<|im_end|>
<|im_start|>user
Write a C++ function that sorts a list of numbers in ascending order. Implement it from scratch without using 
built-in sorting functions. Write your code without any comments or explanations, just the 
implementation.<|im_end|>
<|im_start|>assistant
<code>
void sort(std::vector<int>& numbers) {  
    if (numbers.size() <= 1) return;  
    std::vector<int> left(numbers.begin(), numbers.begin() + numbers.size() / 2);  
    std::vector<int> right(numbers.begin() + numbers.size() / 2, numbers.end());  
    sort(left);  
    sort(right);  
    int i = 0, j = 0, k = 0;  
    while (i < left.size() && j < right.size()) {  
        if (left[i] <= right[j]) numbers[k++] = left[i++];  
        else numbers[k++] = right[j++];  
    }  
    while (i < left.size()) numbers[k++] = left[i++];  
    while (j < right.size()) numbers[k++] = right[j++];  
}
</code>